# Part 19: Production Deployment with Modal & Autonomous Agents

> Deploying LLM services to cloud infrastructure with Modal, running open-source models (Llama 3), and building autonomous multi-agent systems.

---


## 19.1 What is Modal?

Modal is a serverless cloud platform optimized for ML workloads:
- **No server management** — spin up GPU containers on demand
- **Pay per second** — only pay when running
- **GPU access** — A10G, A100, H100 available
- **Scale to zero** — no idle costs


In [ ]:
# Install: pip install modal
# Setup:   modal setup (one-time authentication)

import modal

app = modal.App("genai-llm-service")

# Define a simple function to run in the cloud
@app.function()
def hello_modal():
    return "Hello from Modal cloud!"

# Run locally (for testing)
if __name__ == "__main__":
    with app.run():
        result = hello_modal.remote()
        print(result)


## 19.2 Modal Image — Dependencies

In [ ]:
# Define the container image with all dependencies
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "transformers",
    "torch",
    "accelerate",
    "huggingface_hub",
    "openai"
)

@app.function(image=image)
def run_with_deps():
    import torch
    return f"PyTorch version: {torch.__version__}"


## 19.3 Running Llama 3 on Modal GPU

In [ ]:
import modal

# HuggingFace token stored as Modal secret
HF_SECRET = modal.Secret.from_name("huggingface-secret")

llm_image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "transformers", "torch", "accelerate", "huggingface_hub"
)

@app.cls(
    image=llm_image,
    gpu="A10G",           # GPU type: T4, A10G, A100, H100
    secrets=[HF_SECRET],
    timeout=600,
    container_idle_timeout=300
)
class LlamaService:
    """A GPU-backed Llama 3 inference service."""
    
    @modal.enter()
    def load_model(self):
        """Load model once at container start."""
        from transformers import AutoTokenizer, AutoModelForCausalLM
        import torch
        
        model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        print("Model loaded!")
    
    @modal.method()
    def generate(self, prompt: str, max_new_tokens: int = 256) -> str:
        import torch
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response[len(prompt):]  # strip the prompt


## 19.4 Modal Web Endpoint (REST API)

In [ ]:
from pydantic import BaseModel as PydanticBase

class PricingRequest(PydanticBase):
    title: str
    description: str = ""
    category: str = ""

class PricingResponse(PydanticBase):
    estimated_price: float
    confidence: str
    reasoning: str

@app.function(image=llm_image, gpu="A10G", secrets=[HF_SECRET])
@modal.web_endpoint(method="POST")
def price_estimator(request: PricingRequest) -> PricingResponse:
    """REST endpoint for product price estimation."""
    # In production: load model from class above
    from openai import OpenAI
    client = OpenAI()
    
    prompt = f"Title: {request.title}\nCategory: {request.category}"
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Estimate product price. Respond with JSON: {price, confidence, reasoning}"},
            {"role": "user",   "content": prompt}
        ],
        response_format={"type": "json_object"}
    )
    
    import json
    data = json.loads(response.choices[0].message.content)
    return PricingResponse(
        estimated_price=data.get("price", 0),
        confidence=data.get("confidence", "medium"),
        reasoning=data.get("reasoning", "")
    )


## 19.5 Pricer Service Pattern (Production ML)

In [ ]:
# pricer_service.py — production-ready pricing service

from dataclasses import dataclass
from openai import OpenAI
import re

@dataclass
class PricerService:
    """Multi-model price estimation with ensemble."""
    openai_model: str = "ft:gpt-4o-mini:..."  # your fine-tuned model
    fallback_model: str = "gpt-4o-mini"
    
    def __post_init__(self):
        self.client = OpenAI()
    
    def estimate(self, title: str, description: str = "", category: str = "") -> float:
        """Estimate price using fine-tuned model with fallback."""
        prompt = f"Title: {title}\nCategory: {category}\nDescription: {description[:200]}"
        
        for model in [self.openai_model, self.fallback_model]:
            try:
                response = self.client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": "Estimate product price."},
                        {"role": "user",   "content": prompt}
                    ]
                )
                text = response.choices[0].message.content
                match = re.search(r'\$([\d.]+)', text)
                if match:
                    return float(match.group(1))
            except Exception as e:
                print(f"Model {model} failed: {e}")
        
        return -1.0  # failed to estimate


## 19.6 Autonomous Multi-Agent System

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

class SpecialistAgent:
    """A specialist agent that can use tools."""
    
    def __init__(self, name: str, system_prompt: str, tools: list = None):
        self.name = name
        self.system_prompt = system_prompt
        self.tools = tools or []
    
    def run(self, task: str, context: dict = None) -> str:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user",   "content": f"Task: {task}\nContext: {json.dumps(context or {})}"}
        ]
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=self.tools if self.tools else None
        )
        return response.choices[0].message.content

class OrchestratorAgent:
    """Orchestrates multiple specialist agents."""
    
    def __init__(self, specialists: list[SpecialistAgent]):
        self.specialists = {agent.name: agent for agent in specialists}
        self.client = OpenAI()
    
    def route_and_execute(self, task: str) -> str:
        # Ask orchestrator which specialist to use
        routing_response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": f"""Which specialist should handle this task?
Task: {task}
Specialists: {list(self.specialists.keys())}
Reply with just the specialist name."""
            }]
        )
        
        specialist_name = routing_response.choices[0].message.content.strip()
        if specialist_name in self.specialists:
            return self.specialists[specialist_name].run(task)
        return "No suitable specialist found"

# Example: Price Research System
researcher = SpecialistAgent(
    name="researcher",
    system_prompt="You research product prices from market data."
)
analyst = SpecialistAgent(
    name="analyst",
    system_prompt="You analyze pricing patterns and suggest optimal prices."
)
writer = SpecialistAgent(
    name="writer",
    system_prompt="You write clear pricing reports."
)

orchestrator = OrchestratorAgent([researcher, analyst, writer])


## 19.7 Deployment Lifecycle

```
Development → Testing → Staging → Production

Local testing:    modal run service.py
Deploy to Modal:  modal deploy service.py
Monitor:          modal app logs genai-llm-service
Rollback:         modal app stop genai-llm-service
```

### Ephemeral vs Deployed Apps

| Mode | Command | Lifecycle |
|------|---------|-----------|
| Ephemeral | `modal run` | Runs, then stops |
| Deployed | `modal deploy` | Runs until explicitly stopped |
| Serve (dev) | `modal serve` | Redeploys on code change |


In [ ]:
# Keep-warm pattern — prevent cold starts
# keep_warm.py
import modal
import time

app = modal.App("pricer-keepwarm")

@app.function(schedule=modal.Period(minutes=5))
def keep_warm():
    """Ping the service every 5 minutes to prevent cold starts."""
    import requests
    response = requests.post(
        "https://your-modal-endpoint.modal.run",
        json={"title": "test", "category": "test"},
        timeout=30
    )
    print(f"Keep-warm ping: {response.status_code}")


## 19.8 Summary

| Concept | Details |
|---------|---------|
| Modal App | `modal.App("name")` |
| Container image | `modal.Image.debian_slim().pip_install(...)` |
| GPU function | `@app.function(gpu="A10G")` |
| Class (stateful) | `@app.cls(gpu=...)` with `@modal.enter()` |
| Web endpoint | `@modal.web_endpoint(method="POST")` |
| Secrets | `modal.Secret.from_name("key-name")` |
| Schedule | `schedule=modal.Period(minutes=5)` |

---

**Next:** [Part 20 — Agentic AI Foundations & OpenAI Agents SDK](Part20_Agentic_AI_Foundations.ipynb)
